In [6]:
# Mount your Google Drive so downloaded/processed data persists across sessions.
# You'll be prompted to authorize access — click the link, sign in, paste the code.
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/esm2-variant-benchmark'
os.makedirs(f'{PROJECT_DIR}/data/raw', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data/processed', exist_ok=True)
print(f'Project folder ready at: {PROJECT_DIR}')

Mounted at /content/drive
Project folder ready at: /content/drive/MyDrive/esm2-variant-benchmark


In [7]:
# Install/import everything this notebook needs.
# biopython: sequence handling | requests: API calls to ClinVar/UniProt
# pandas: tabular data | tqdm: progress bars for long loops
!pip install -q biopython requests pandas tqdm

import re
import requests
import pandas as pd
from tqdm import tqdm

In [8]:
# List the genes you want variants for. Start with 20-30 well-studied genes
# (better ClinVar annotation quality = cleaner labels later).
# Edit this list to your actual protein panel before running the rest.
TARGET_GENES = [
    'TP53', 'BRCA1', 'BRCA2', 'PTEN', 'EGFR', 'KRAS', 'BRAF', 'MYC',
    'CFTR', 'LDLR', 'APOE', 'MLH1', 'MSH2', 'VHL', 'RB1', 'ATM',
    'PAH', 'HBB', 'FBN1', 'COL1A1'
]

print(f'{len(TARGET_GENES)} genes in panel')

20 genes in panel


In [9]:
# Download NCBI's ClinVar variant summary file (a few hundred MB, compressed).
# Saved into Drive so you only need to do this once across your whole project.
import os

CLINVAR_URL = 'https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz'
local_gz = f'{PROJECT_DIR}/data/raw/variant_summary.txt.gz'

if not os.path.exists(local_gz):
    print('Downloading ClinVar variant summary, this can take a few minutes...')
    r = requests.get(CLINVAR_URL, stream=True)
    with open(local_gz, 'wb') as f:
        for chunk in tqdm(r.iter_content(chunk_size=8192)):
            f.write(chunk)
else:
    print('Already downloaded, skipping.')

53936it [00:24, 2223.97it/s]


In [10]:
# The full file covers every gene in ClinVar (millions of rows).
# Read it in chunks (it's too big for memory at once) and keep only rows
# matching your gene panel, single nucleotide variants, on GRCh38.
usecols = ['GeneSymbol', 'ClinicalSignificance', 'Type', 'Assembly',
           'Name', 'ReviewStatus', 'PhenotypeList']

chunks = pd.read_csv(local_gz, sep='\t', usecols=usecols,
                      chunksize=200_000, low_memory=False)

filtered_chunks = []
for chunk in tqdm(chunks):
    sub = chunk[
        chunk['GeneSymbol'].isin(TARGET_GENES) &
        (chunk['Type'] == 'single nucleotide variant') &
        (chunk['Assembly'] == 'GRCh38')
    ]
    filtered_chunks.append(sub)

clinvar_df = pd.concat(filtered_chunks, ignore_index=True)
print(f'{len(clinvar_df)} candidate rows after gene/type/assembly filter')

46it [00:42,  1.07it/s]

94954 candidate rows after gene/type/assembly filter


In [11]:
# ClinVar stores the protein change inside a text field, e.g.
# "NM_000546.6(TP53):c.215C>G (p.Pro72Arg)" — pull out just "Pro72Arg".
# Drop synonymous changes (same amino acid before and after) and keep
# only variants with an unambiguous Pathogenic/Benign label (drop VUS etc).
aa3 = 'Ala|Arg|Asn|Asp|Cys|Gln|Glu|Gly|His|Ile|Leu|Lys|Met|Phe|Pro|Ser|Thr|Trp|Tyr|Val'
protein_change_re = re.compile(rf'p\.(({aa3})(\d+)({aa3}))')

def extract_protein_change(name):
    match = protein_change_re.search(str(name))
    if match:
        return match.group(1)
    return None

clinvar_df['protein_change'] = clinvar_df['Name'].apply(extract_protein_change)
clinvar_df = clinvar_df.dropna(subset=['protein_change'])

clinvar_df = clinvar_df[
    clinvar_df['protein_change'].str[:3] != clinvar_df['protein_change'].str[-3:]
]

sig_map = {
    'Pathogenic': 1, 'Likely pathogenic': 1,
    'Benign': 0, 'Likely benign': 0
}
clinvar_df['label'] = clinvar_df['ClinicalSignificance'].map(sig_map)
clinvar_df = clinvar_df.dropna(subset=['label'])
clinvar_df['label'] = clinvar_df['label'].astype(int)

print(clinvar_df['label'].value_counts())
clinvar_df.head()

label
1    4164
0    2090
Name: count, dtype: int64


,Type,Name,GeneSymbol,ClinicalSignificance,PhenotypeList,Assembly,ReviewStatus,protein_change,label
1,single nucleotide variant,NM_000277.3(PAH):c.1222C>T (p.Arg408Trp),PAH,Pathogenic,Phenylketonuria|not provided|Inborn genetic di...,GRCh38,reviewed by expert panel,Arg408Trp,1
2,single nucleotide variant,NM_000277.3(PAH):c.932T>C (p.Leu311Pro),PAH,Pathogenic,Phenylketonuria|not provided,GRCh38,reviewed by expert panel,Leu311Pro,1
4,single nucleotide variant,NM_000277.3(PAH):c.838G>A (p.Glu280Lys),PAH,Pathogenic,"Phenylketonuria|not provided|Polymicrogyria, p...",GRCh38,"criteria provided, multiple submitters, no con...",Glu280Lys,1
6,single nucleotide variant,NM_000277.3(PAH):c.782G>A (p.Arg261Gln),PAH,Pathogenic,Phenylketonuria|not provided|PAH-related disorder,GRCh38,reviewed by expert panel,Arg261Gln,1
7,single nucleotide variant,NM_000277.3(PAH):c.261C>A (p.Ser87Arg),PAH,Pathogenic,Hyperphenylalaninemia|not provided|Phenylketon...,GRCh38,reviewed by expert panel,Ser87Arg,1


In [12]:
# For each gene, query UniProt's REST API for the reviewed (Swiss-Prot)
# human entry — this is the "canonical" protein we'll use for sequences,
# conservation scoring, and ESM-2 later.
def get_uniprot_accession(gene_symbol):
    url = 'https://rest.uniprot.org/uniprotkb/search'
    params = {
        'query': f'gene:{gene_symbol} AND organism_id:9606 AND reviewed:true',
        'fields': 'accession,gene_names',
        'format': 'json',
        'size': 1
    }
    r = requests.get(url, params=params)
    r.raise_for_status()
    results = r.json().get('results', [])
    if results:
        return results[0]['primaryAccession']
    return None

gene_to_accession = {}
for gene in tqdm(TARGET_GENES):
    acc = get_uniprot_accession(gene)
    gene_to_accession[gene] = acc
    if acc is None:
        print(f'WARNING: no UniProt entry found for {gene}')

gene_to_accession

100%|██████████| 20/20 [00:20<00:00,  1.02s/it]


{'TP53': 'P04637',
 'BRCA1': 'P38398',
 'BRCA2': 'P51587',
 'PTEN': 'P60484',
 'EGFR': 'P00533',
 'KRAS': 'P01116',
 'BRAF': 'P15056',
 'MYC': 'P01106',
 'CFTR': 'P13569',
 'LDLR': 'P01130',
 'APOE': 'P02649',
 'MLH1': 'P40692',
 'MSH2': 'P43246',
 'VHL': 'P40337',
 'RB1': 'P06400',
 'ATM': 'Q13315',
 'PAH': 'P00439',
 'HBB': 'P68871',
 'FBN1': 'P35555',
 'COL1A1': 'P02452'}

In [13]:
# Pull the actual amino acid sequence for each UniProt accession found above.
# These sequences are what WP2 (conservation) and WP3 (ESM-2) will both use.
def fetch_fasta(accession):
    url = f'https://rest.uniprot.org/uniprotkb/{accession}.fasta'
    r = requests.get(url)
    r.raise_for_status()
    return r.text

fasta_records = []
for gene, acc in tqdm(gene_to_accession.items()):
    if acc is None:
        continue
    fasta_records.append(fetch_fasta(acc))

fasta_path = f'{PROJECT_DIR}/data/raw/sequences.fasta'
with open(fasta_path, 'w') as f:
    f.write('\n'.join(fasta_records))

print(f'Wrote {len(fasta_records)} sequences to {fasta_path}')

100%|██████████| 20/20 [00:18<00:00,  1.07it/s]

Wrote 20 sequences to /content/drive/MyDrive/esm2-variant-benchmark/data/raw/sequences.fasta


In [16]:
# Join each variant row to its protein's UniProt accession, drop any gene
# that failed to resolve, then save the final clean variant table.
clinvar_df['uniprot_accession'] = clinvar_df['GeneSymbol'].map(gene_to_accession)
clinvar_df = clinvar_df.dropna(subset=['uniprot_accession'])

final_cols = ['GeneSymbol', 'uniprot_accession', 'protein_change',
              'label', 'ReviewStatus', 'PhenotypeList']
clinvar_df = clinvar_df[final_cols].drop_duplicates()

variants_path = f'{PROJECT_DIR}/data/processed/clinvar_variants.tsv'
clinvar_df.to_csv(variants_path, sep='\t', index=False)

print(f'Final dataset: {len(clinvar_df)} variants across {clinvar_df["GeneSymbol"].nunique()} genes')
print(f'Saved to {variants_path}')
clinvar_df['label'].value_counts()

# Drop genes with unusable class imbalance (too few of one label)
DROP_GENES = ['MYC', 'KRAS', 'PAH']
clinvar_df = clinvar_df[~clinvar_df['GeneSymbol'].isin(DROP_GENES)]

print('Updated variants per gene:')
print(clinvar_df.groupby('GeneSymbol')['label'].value_counts().unstack(fill_value=0))
print(f'\nTotal: {len(clinvar_df)} variants across {clinvar_df["GeneSymbol"].nunique()} genes')

Final dataset: 6238 variants across 20 genes
Saved to /content/drive/MyDrive/esm2-variant-benchmark/data/processed/clinvar_variants.tsv
Updated variants per gene:
label         0     1
GeneSymbol           
APOE          8    11
ATM         148    70
BRAF         11    85
BRCA1       441   175
BRCA2       686    83
CFTR         13   224
COL1A1       72   438
EGFR         39    13
FBN1         55  1196
HBB          16    71
LDLR         38   504
MLH1         36   165
MSH2        285   109
PTEN          9   207
RB1          36    32
TP53        155   119
VHL          35   122

Total: 5707 variants across 17 genes


In [15]:
# Check class balance and gene coverage before trusting this data downstream.
# Ideally no gene is missing one label entirely, and overall balance isn't
# worse than roughly 80/20 — if it is, consider expanding the gene panel.
print('Variants per gene:')
print(clinvar_df.groupby('GeneSymbol')['label'].value_counts().unstack(fill_value=0))

Variants per gene:
label         0     1
GeneSymbol           
APOE          8    11
ATM         148    70
BRAF         11    85
BRCA1       441   175
BRCA2       686    83
CFTR         13   224
COL1A1       72   438
EGFR         39    13
FBN1         55  1196
HBB          16    71
KRAS          1    56
LDLR         38   504
MLH1         36   165
MSH2        285   109
MYC           0     4
PAH           1   469
PTEN          9   207
RB1          36    32
TP53        155   119
VHL          35   122
